In [1]:
# Installing all the Python packages I need for this project.
# torch and torchvision = deep learning framework and image tools
# pandas = working with tabular data (CSVs)
# geopandas = pandas but for geographic/spatial data
# Pillow = opening and handling image files
# requests = making HTTP calls to APIs
# shapely = geometry operations (points, polygons)
# tqdm = progress bars so I can see how far along a loop is
# osmnx = downloading road networks from OpenStreetMap
!pip install torch torchvision pandas geopandas Pillow requests shapely tqdm numpy matplotlib seaborn scikit-learn osmnx

In [2]:
# Importing the specific tools I need from each installed package.
# I don't import whole packages when I only need specific parts — it's cleaner.

import torch                              # Core PyTorch library for tensor operations and GPU support
import torch.nn as nn                    # Neural network building blocks (layers, loss functions)
import torchvision.models as models      # Pre-trained models like ResNet-18
import torchvision.transforms as transforms  # Image preprocessing pipelines
from torch.utils.data import Dataset, DataLoader  # Custom dataset class + batching utility
import pandas as pd                      # For loading and manipulating my CSV data
import numpy as np                       # Numerical operations, random seeds, array maths
from PIL import Image                    # Opening image files
import os                                # File path operations (checking if files exist etc.)
from tqdm import tqdm                    # Wraps any loop to show a live progress bar

In [10]:
# Defining all my file paths in one place at the top.
# This way if I move a folder, I only change one line instead of hunting
# through the whole notebook. base is the root of my project.
base       = '.\\data'  # update this to point to your local data directory
data_csv   = f'{base}\\Cleaned\\final_data.csv'
image_dir  = f'{base}\\Raw\\images_Chinese'
model_save = f'{base}\\Outputs\\Models\\best_model.pth'


In [11]:
# Loading the Place Pulse dataset and inspecting it.
# final_data.csv has 1.2 million rows — each row is one human pairwise judgment.
# I print category counts to see how many comparisons exist per perception dimension
# and winner counts to check the balance between left/right/equal outcomes.
df=pd.read_csv(data_csv)
print(f'total rows:', len(df))
print(df['category'].value_counts # How many comparisons per dimension (wealthy, safety etc.)
print(df['winner'].value_counts()) # How often left vs right vs equal won

total rows: 1208808
category
safety        364366
lively        264107
beautiful     173249
wealthy       150459
depressing    130793
boring        125834
Name: count, dtype: int64
winner
right    532298
left     516889
equal    159621
Name: count, dtype: int64


In [12]:
# Filtering down to only the 'wealthy' perception dimension.
# I'm using wealthy because it's most directly relevant to my economic outcomes
# (house prices and claimant rates). I also drop 'equal' rows because the model
# needs a clear winner to learn from — ties don't give it a training signal.
df_wealthy=df[df['category']== 'wealthy'].copy()
df_wealthy=df_wealthy[df_wealthy['winner']!='equal'].copy()
df_wealthy=df_wealthy.reset_index(drop=True)  # Resets row numbering from 0 after filtering
print("Wealthy comparisons (no equals):", len(df_wealthy))

Wealthy comparisons (no equals): 130252


In [13]:
# Before training I need to verify the images are actually on disk and accessible.
# I take the first 10 image IDs from the left column and check each one exists.
# If any say MISSING, there's a path problem before I waste time on training.
from os.path import exists
sample_ids = df_wealthy['left_id'].head(10).tolist()
for pid in sample_ids:
  path = f'{image_dir}/{pid}.jpg'
  exists=os.path.exists(path)
  print(f"{pid}.jpg - {'FOUND' if exists else 'MISSING'}")

513d2dbefdc9f03587002515.jpg - FOUND
514145e8fdc9f049260066b8.jpg - FOUND
513da066fdc9f0358700897b.jpg - FOUND
5141355bfdc9f049260049a4.jpg - FOUND
50f5eb17fdc9f065f0007f52.jpg - FOUND
5140da4dfdc9f04926003dab.jpg - FOUND
51409c68fdc9f049260011ba.jpg - FOUND
51409c68fdc9f049260011ba.jpg - FOUND
513d69dafdc9f03587004892.jpg - FOUND
513d69dafdc9f03587004892.jpg - FOUND


In [ ]:
################### TRAINING SIAMESE CNN ####################

In [8]:
#First, I convert the winner column into a numeric label
df_wealthy['label']=df_wealthy['winner'].map({'left':1, 'right': 0})
#Where 1 means left won (looks wealthier) and 0 means right image won(looks wealthier).

In [9]:
# Splitting data into training, validation and test sets.

# IMPORTANT: I split by IMAGE ID, not by row. If I split by row, the same image
# could appear in both train and test sets (in different pairs), which would
# let the model "cheat" by memorising images it already saw — this is data leakage.
# Splitting by image ID ensures each image only ever appears in one split.
# 80% train / 10% validation / 10% test is a standard split.
# np.random.seed(42) makes the split reproducible — same result every time I run it.

all_ids= pd.unique(df_wealthy[['left_id', 'right_id']].values.ravel())
np.random.seed(42)
np.random.shuffle(all_ids) # Just randomly shuffling image IDs before splitting

n= len(all_ids)
train_ids= set(all_ids[:int(0.8*n)])  # First 80% of image IDs
val_ids= set(all_ids[int(0.8*n):int(0.9*n)])  # Next 10%
test_ids= set(all_ids[int(0.9*n):])  # Final 10%

# Only keep rows where BOTH images in the pair belong to the same split
train_df = df_wealthy[df_wealthy['left_id'].isin(train_ids) & df_wealthy['right_id']. isin(train_ids)].reset_index(drop=True)
val_df= df_wealthy[df_wealthy['left_id'].isin(val_ids) & df_wealthy['right_id'].isin(val_ids)].reset_index(drop=True)
test_df= df_wealthy[df_wealthy['left_id'].isin(test_ids) & df_wealthy['right_id'].isin(test_ids)].reset_index(drop=True)

print("Train:", len(train_df), "| Val:", len(val_df), "| Test:", len(test_df))

Train: 83159 | Val: 1328 | Test: 1284


In [10]:
# Defining the image preprocessing pipeline.
# Every image must go through these three steps before entering the model:
# 1. Resize to 224x224 — ResNet-18 expects this exact input size
# 2. ToTensor — converts the image from a PIL object to a PyTorch tensor (a 3D array
#    of pixel values between 0 and 1)
# 3. Normalize — subtracts the ImageNet mean and divides by ImageNet std dev.
#    This is required because ResNet-18 was pre-trained on ImageNet with these
#    exact statistics — normalising my images to match helps transfer learning work
transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

In [ ]:
# Defining a custom Dataset class so PyTorch knows how to load my data.
# PyTorch's DataLoader needs a Dataset object that implements three methods:
# __init__: stores the dataframe, image folder path, and transform
# __len__: tells the DataLoader how many items are in the dataset
# __getitem__: given an index, loads the left image, right image, and label for that row
# The load_image helper opens a jpg by panoid, converts to RGB, and applies the transform

class PlacePulseDataset(Dataset):
  def __init__(self, dataframe, image_dir, transform):
    self.df=dataframe
    self.image_dir=image_dir
    self.transform=transform

  def __len__(self):
    return len(self.df)  # Total number of comparison pairs
 
  def load_image(self, panoid):
    # Build the full file path from the panorama ID
    path = os.path.join(self.image_dir, f'{panoid}.jpg')
    img = Image.open(path).convert('RGB') # Always convert to RGB — some images are greyscale
    return self.transform(img) # Apply resize, tensor, normalise

  def __getitem__(self,idx):
    row = self.df.iloc[idx] # Get one row from the dataframe
    left_img = self.load_image(row['left_id']) # Load left image of the pair
    right_img = self.load_image(row['right_id']) # Load right image of the pair
    label = row['label'] # 1 if left won, 0 if right won
    return left_img, right_img, label

In [12]:
# Creating Dataset objects for each split, then wrapping them in DataLoaders.
# DataLoader handles batching (grouping 32 images at a time), shuffling,
# and feeding data to the GPU efficiently.
# shuffle=True for training — randomises order each epoch so model doesn't memorise order
# shuffle=False for val/test — consistent order for fair evaluation
# num_workers=0 — critical on Windows: multiprocessing hangs in Jupyter, so I use 0
# pin_memory=True — speeds up CPU-to-GPU data transfer

train_dataset = PlacePulseDataset(train_df, image_dir, transform)
val_dataset = PlacePulseDataset(val_df, image_dir, transform)
test_dataset = PlacePulseDataset(test_df, image_dir, transform)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle = True, num_workers=0, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle = False, num_workers=0, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle = False, num_workers=0, pin_memory=True)

In [13]:
# Defining the Siamese CNN architecture.
# A Siamese network has two identical branches (sharing the same weights)
# that each process one image from the pair, then compares the outputs.
# I use ResNet-18 as the backbone — a well-established CNN pre-trained on ImageNet.
# I remove its final classification layer (which was for 1000 ImageNet categories)
# and replace it with my own comparison head that outputs a single probability.
class SiameseCNN(nn.Module):
  def __init__(self):
    super().__init__()
    base = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
    # Strip the final fully-connected layer — I want the 512-dim feature vector, not class predictions
    self.encoder = nn.Sequential(*list(base.children())[:-1])
    # My custom comparison head: takes the difference vector and predicts which image looks wealthier
        # Linear(512->256): compress feature space
        # ReLU: introduces non-linearity so the model can learn complex patterns
        # Dropout(0.3): randomly zeros 30% of neurons during training to prevent overfitting
        # Linear(256->1): collapse to single output neuron
        # Sigmoid: squashes output to 0-1 range (interpretable as probability)
    self.fc = nn.Sequential(
        nn.Linear(512, 256),
        nn.ReLU(),
        nn.Dropout(0.3),
        nn.Linear(256, 1),
        nn.Sigmoid()
    )

  def forward_once(self, x):
    # Pass one image through ResNet encoder to get its 512-dimensional embedding
    out= self.encoder(x)
    return out.view(out.size(0), -1) #flatten to (batch, 512) from (batch, 512, 1, 1)

  def forward(self, left_img, right_img):
    feat_left = self.forward_once(left_img) # Encode left image → 512-dim vector
    feat_right = self.forward_once(right_img) # Encode right image → 512-dim vector
    #Calculate the difference in embeddings (captures relative visual features)
    diff = feat_left - feat_right
    return self.fc(diff) # Predict probability that left looks wealthier

In [14]:
# Setting up training components.
# device: checks if a GPU is available — if yes, runs on CUDA (much faster than CPU)
# model.to(device): moves all model parameters onto the GPU
# BCELoss: Binary Cross-Entropy — standard loss for binary classification (left won or right won)
# Adam optimiser: updates model weights after each batch; lr=0.001 controls step size
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print("Using:", device) #whcih should say cuda

model = SiameseCNN().to(device)
criterion = nn.BCELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

Using: cuda


In [ ]:
# -------------- TESTING ----------------------

# Before running the full training loop I test each component individually.
# This saves hours of wasted GPU time if something is broken.

In [15]:
# Test 1: Can I open a single image file from disk?

from PIL import Image
import os

# Just try opening the very first image in your dataset
test_id = train_df['left_id'].iloc[0] # Get the first image ID
test_path = os.path.join(image_dir, f'{test_id}.jpg') # Build its full path

print("Looking for:", test_path)
print("File exists:", os.path.exists(test_path))

img = Image.open(test_path).convert('RGB')
print("Image size:", img.size) #Should be (400,300) for Place Pulse 2.0 images
print("Image mode:", img.mode) # Should be RGB

Looking for: E:\Warwick\EconDS\Dissertation\Data\Raw\images_Chinese\513d2dbefdc9f03587002515.jpg
File exists: True
Image size: (400, 300)
Image mode: RGB


In [17]:
# Test 2: Does the transform pipeline work on that image?
tensor = transform(img)
print("Tensor shape:", tensor.shape) #Should be torch.size(3,224,224)
print("Tensor min/max:", tensor.min().item(), tensor.max().item()) # Should be roughly -2 to 2

Tensor shape: torch.Size([3, 224, 224])
Tensor min/max: -1.9295316934585571 2.640000104904175


In [18]:
# Testing the dataset getitem directly, bypassing DataLoader
# Call __getitem__ manually on index 0 — no DataLoader involved
sample = train_dataset[0]
print("Left image tensor shape:", sample[0].shape)
print("Right image tensor shape:", sample[1].shape)
print("Label:", sample[2]) # 0 or 1

Left image tensor shape: torch.Size([3, 224, 224])
Right image tensor shape: torch.Size([3, 224, 224])
Label: 1


In [ ]:
#### -------------- All three cells worked perfectly. Images load fine, transforms work, dataset indexing works. 
#### -------------- The problem might be specifically the DataLoader with multiple workers - hangs when parallel processes on Windows

## So i went back and altered DataLoader cell num_workers=2 to num_workers=0


In [16]:
# Setting how many full passes through the training data I want.
# Each epoch = the model sees every training pair once.
# I also define the checkpoint path - where to save progress after each epoch
# in case the notebook crashes or disconnects mid-training.
num_epochs = 10
checkpoint = f'{base}\\Outputs\\Models\\checkpoint_wealthy.pth'

In [17]:
# The main training loop. For each epoch:
# TRAINING PHASE: model sees training pairs, makes predictions, calculates loss,
#backpropagates the error, and updates weights
# VALIDATION PHASE: model sees validation pairs but weights are NOT updated —
#this tells me how well it generalises to unseen data without overfitting
# After each epoch: prints metrics and saves a checkpoint to computer

best_val_acc = 0.0  # tracker for best epoch

for epoch in range(num_epochs):

    # -------- Training --------
    model.train() # Puts model in training mode - activates Dropout
    train_loss, train_correct, train_total = 0, 0, 0

    for left_img, right_img, label in tqdm(train_loader, desc=f'Epoch {epoch+1} Train'):
        left_img  = left_img.to(device) # Move batch to GPU
        right_img = right_img.to(device)
        label     = label.to(device).float().unsqueeze(1)   # shape: (batch, 1) to match model output

        optimizer.zero_grad()     # Clear gradients from previous batch
        output = model(left_img, right_img) # Forward pass: get predictions
        loss   = criterion(output, label) # Compare predictions to true labels
        loss.backward()   # Backpropagate: compute gradients
        optimizer.step() # Update weights using those gradients

        train_loss    += loss.item() # Accumulate loss for this epoch
        preds          = (output > 0.5).float() # Convert probabilities to binary predictions
        train_correct += (preds == label).sum().item() # Count correct predictions
        train_total   += label.size(0) # Count total predictions

    # -------- Validation --------
    model.eval() # Puts model in evaluation mode - deactivates Dropout
    val_loss, val_correct, val_total = 0, 0, 0

    with torch.no_grad(): # Tells PyTorch not to compute gradients - saves memory and speeds up
        for left_img, right_img, label in val_loader:
            left_img  = left_img.to(device)
            right_img = right_img.to(device)
            label     = label.to(device).float().unsqueeze(1)  # shape: (batch, 1)

            output = model(left_img, right_img)
            loss   = criterion(output, label)

            val_loss    += loss.item()
            preds        = (output > 0.5).float()
            val_correct += (preds == label).sum().item()
            val_total   += label.size(0)

    # -------- Epoch Summary --------
    train_acc = train_correct / train_total
    val_acc   = val_correct   / val_total

    print(f"Epoch {epoch+1}/{num_epochs} | "
          f"Train Loss: {train_loss/len(train_loader):.4f} | Train Acc: {train_acc:.4f} | "
          f"Val Loss: {val_loss/len(val_loader):.4f} | Val Acc: {val_acc:.4f}")

    # -------- Save Checkpoint --------
    # Save every epoch (for resuming if it crashes)
    torch.save({
        'epoch': epoch + 1,
        'model_state_dict': model.state_dict(), #The actual learned weights
        'optimizer_state_dict': optimizer.state_dict(), #Optimizer state for resuming
        'val_acc': val_acc
    }, checkpoint)

# Also save separately if this is the best val acc so far
 # I use val_acc not train_acc because train_acc keeps climbing even when overfitting
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        best_model_path = f'{base}\\Outputs\\Models\\best_model.pth'
        torch.save(model.state_dict(), best_model_path)
        print(f"  *** New best model saved (epoch {epoch+1}, val_acc={val_acc:.4f}) ***")
    else:
        print(f"  Checkpoint saved (epoch {epoch+1})")
# Result: best model was epoch 6, val_acc = 0.6506

Epoch 1 Train: 100%|██████████| 2599/2599 [26:57<00:00,  1.61it/s]


Epoch 1/10 | Train Loss: 0.6529 | Train Acc: 0.6211 | Val Loss: 0.6584 | Val Acc: 0.6032
  *** New best model saved (epoch 1, val_acc=0.6032) ***


Epoch 2 Train: 100%|██████████| 2599/2599 [13:51<00:00,  3.13it/s]


Epoch 2/10 | Train Loss: 0.6354 | Train Acc: 0.6407 | Val Loss: 0.6502 | Val Acc: 0.6099
  *** New best model saved (epoch 2, val_acc=0.6099) ***


Epoch 3 Train: 100%|██████████| 2599/2599 [13:51<00:00,  3.13it/s]


...

Epoch 8/10 | Train Loss: 0.5335 | Train Acc: 0.7332 | Val Loss: 0.6808 | Val Acc: 0.6370
  Checkpoint saved (epoch 8)


Epoch 9 Train: 100%|██████████| 2599/2599 [13:52<00:00,  3.12it/s]


Epoch 9/10 | Train Loss: 0.4904 | Train Acc: 0.7645 | Val Loss: 0.7122 | Val Acc: 0.6340
  Checkpoint saved (epoch 9)


Epoch 10 Train: 100%|██████████| 2599/2599 [13:54<00:00,  3.12it/s]


Epoch 10/10 | Train Loss: 0.4469 | Train Acc: 0.7921 | Val Loss: 0.7144 | Val Acc: 0.6370
  Checkpoint saved (epoch 10)


In [ ]:
best_val_acc #Epoch 6